## Differential Solar Rotation 
### Task 3

Cortés, I., (2024) for the AstroLab Summer term, Heidelberg University

iliana.cortes@h-its.org


## Introduction

In this notebook, we will explore some key concepts related to solar observatories, Lagrangian points, and the structure of the Sun. 

### Questions

1. **What are the Lagrangian points? Why was SOHO put onto L1?**

   - Lagrangian points (or libration points) are **equilibrium points** where the gravitational forces of two massive bodies (like the Earth and the Sun) and the centrifugal force balance. Small objects can remain stationary relative to the two larger bodies at these points. 

2. **Why was SOHO put in a halo orbit around L1 instead of directly at L1?**

   - SOHO was placed in a halo orbit around L1 to maintain continuous, stable observation of the Sun while minimizing the need for propulsion. Unlike a direct placement at L1, a halo orbit provides a **stable, quasi-periodic trajectory** that reduces the need for active station-keeping.

   - **Reference**: [Halo Orbits Around Lagrangian Points](https://www.fossilhunters.xyz/spaceflight/halo-orbits-around-lagrangian-points.html) (external link for further reading)

   - **Types of Orbits Around Lagrangian Points**:
     - **Lissajous Orbit**: A quasi-periodic orbit around a Lagrangian point, named after Jules Antoine Lissajous. These orbits combine components in the plane of the two primary bodies and components perpendicular to this plane.
     - **Lyapunov Orbit**: A curved path in the plane of the two primary bodies, entirely two-dimensional.
     - **Halo Orbit**: A periodic, three-dimensional orbit around a Lagrangian point.

3. **What distinguishes the three outer layers of the Sun—the photosphere, chromosphere, and corona—from each other?**

   - **Temperature**: 
     - The photosphere is the coolest (~5,500°C).
     - The chromosphere becomes progressively hotter.
     - The corona reaches extreme temperatures (~1-3 million °C).
   - **Visibility**:
     - The photosphere is the visible "surface" of the Sun.
     - The chromosphere and corona are visible only during eclipses or with special instruments (e.g., ultraviolet or X-rays).
   - **Density**:
     - The photosphere is the densest, followed by the chromosphere.
     - The corona is the least dense and most diffuse.
   - **Activity**:
     - The chromosphere shows dynamic phenomena like **spicules** and **prominences**.
     - The corona hosts high-energy events such as **solar flares** and the **solar wind**.




   
<div style="text-align: center;">
    <img src="https://pwonlyias.com/wp-content/uploads/2023/11/picture9-655865a99d1a6.webp" alt="Solar Layers" width="400"/>
</div>

4. **Why is the temperature gradient in the Solar Corona positive (i.e., the farther out, the higher the temperature)?**

   The positive temperature gradient in the corona is a well-known puzzle in solar physics, termed the **coronal heating problem**. Despite being farther from the Sun’s core, the corona reaches temperatures of millions of degrees. This is counterintuitive, as one might expect temperatures to decrease with distance.

   - **Magnetic Fields**: The Sun's magnetic fields transport energy outward through waves and reconnection events.
   - **Wave Heating**: Magnetic and acoustic waves transfer energy upwards and release it in the corona.
   - **Nanoflares**: Small-scale bursts of energy from magnetic reconnection may contribute to localized heating.
   - **Low Density**: The corona’s extremely low density makes traditional heat conduction inefficient, allowing heat to accumulate.
   - **Energy Dissipation**: The interaction of waves and magnetic fields in the corona results in the conversion of magnetic energy into heat.

By understanding these concepts, we gain insight into why solar observatories like SOHO are strategically positioned and how the solar atmosphere functions at various layers.


## Determination of the differential solar rotation using SOHO data


In [1]:
# Libraries

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as ct
from astropy import units as u


### Heliographic Coordinates 
Adapted from [CESAR's teacher guide](https://cesar.esa.int/upload/201710/teachers_guide_sun_33.pdf) and [Heliographic coordinates](https://cesar.esa.int/upload/201712/heliographic_coordinates_booklet_248.pdf)



Sun-Earth system, where:
- $\alpha$: angular diameter of the Sun
- $\rho$: angle between the sunspot and the visual

<div style="text-align: center;">
    <!--img src="https://i.ibb.co/vvCsBjyy/sunspot.png" width="600"/-->
    <img src="https://i.ibb.co/CpxJvLKg/sunspot.png" width="600"/>
    
</div>


Solar coordinates, where
- $B$: solar latitude
- $L$: solar longitude
- -$P_m$: position angle
- $P$: Is the position angle of the solar rotation axis measured from the N-S direction. Positive when it’s to the East and negative when it’s to the West.
- $𝐵_0$: Heliographic latitude of the solar disk center.
- $𝐿_0$: Heliographic longitude of the solar disk center

The values $\alpha, P, B_0, L_0$ are to be retrieved from a database. E.g. [Solar Ephemeris](https://www.kso.ac.at/beobachtungen/ephemeris_en.php?date=2005-09-01&time=11%3A44%3A59&lat=46.67747&lon=13.90172)


<div style="text-align: center;">
    <!--img src="https://i.ibb.co/vvCsBjyy/sunspot.png" width="600"/-->
    <img src="https://i.ibb.co/GvXqWs0L/coords.png" width="600"/>
    
</div>



- $(X,Y)$: position of the sunspot in image coordinates
- $R$: radius of the Sun in image coordinates

Following the notation from [CESAR's teacher guide](https://cesar.esa.int/upload/201710/teachers_guide_sun_33.pdf), we have:

$$P_m = \arctan\left(\frac{X}{Y}\right)$$


$$R_m = \sqrt{X^2+Y^2}$$

And the angle between the visual and the sunspot:

$$\rho = \arcsin\left(\frac{R_m}{R}\right) - \frac{\alpha}{2}\frac{R_m}{R} $$

see [Heliographic coordinates](https://cesar.esa.int/upload/201712/heliographic_coordinates_booklet_248.pdf) for a detailed calculation of $\rho$.

The heliographic coordinates (see [Spherical Trigonometry CESAR's booklet](https://cesar.esa.int/upload/201712/spherical_trigonometry_booklet_554.pdf)) are given by:


- Heliographic longitude $$ \sin(B) = \sin(B_0)\cos(\rho) + \cos(B_0)\sin(\rho)\cos(P-P_m) $$
- Heliographic latitude $$ \sin(L-L0) = \sin(P-P_m)\sin(\rho)\cos(B) $$





### Data acquisition

The first step is to extract the position $(X,Y)$ of the sunspot, taking into account the center of the Sun in image coordinates, as well as the radius of the Sun. The procedure is described in the task. 

In [90]:
data_09_01_05 = {
    'OBS_TIME': ['2005-01-09T06:23:32.198Z', '2005-01-09T11:11:32.200Z', '2005-01-09T15:59:32.202Z', '2005-01-09T23:59:32.206Z'], # observation time
    'POS_X': [75, 87, 99, 121], # x coordinate of the sunspot 
    'POS_Y': [469, 471, 472, 476], # y coordinate of the sunspot
    'SUN_X': [5.127242431641E+02, 5.127218017578E+02, 5.125944824219E+02, 5.126706237793E+02], # x coordinate of the solar centre
    'SUN_Y': [5.119660949707E+02, 5.119720153809E+02, 5.119750671387E+02, 5.119614868164E+02], # y coordinate of the solar centre
    'RAD_SUN': [4.956781079938E+02, 4.956769024329E+02, 4.956756248224E+02, 4.956733357482E+02], # radius of the sun
    # Solar ephemeris
    'ALPHA': [1952.36, 1952.35, 1952.33, 1952.3], # angular diameter of the Sun arcsec
    'P_ANGLE': [-1.999, -2.095, -2.190, -2.350], #position angle of the rotation axis
    'B_0': [-3.969, -3.991, -4.012, -4.048], # latitude of the solar disk centre
    'L_0': [265.014, 262.380, 259.747, 255.357] # longitude of the solar disk centre
}


data_09_01_05 = pd.DataFrame(data_09_01_05)

data_09_01_05['OBS_TIME'] = pd.to_datetime(data_09_01_05['OBS_TIME'])


print(data_09_01_05)

                          OBS_TIME  POS_X  POS_Y       SUN_X       SUN_Y  \
0 2005-01-09 06:23:32.198000+00:00     75    469  512.724243  511.966095   
1 2005-01-09 11:11:32.200000+00:00     87    471  512.721802  511.972015   
2 2005-01-09 15:59:32.202000+00:00     99    472  512.594482  511.975067   
3 2005-01-09 23:59:32.206000+00:00    121    476  512.670624  511.961487   

      RAD_SUN    ALPHA  P_ANGLE    B_0      L_0  
0  495.678108  1952.36   -1.999 -3.969  265.014  
1  495.676902  1952.35   -2.095 -3.991  262.380  
2  495.675625  1952.33   -2.190 -4.012  259.747  
3  495.673336  1952.30   -2.350 -4.048  255.357  


### Heliographic coordinates calculation

In [349]:

def helio_coordinates(row):
    R_sun = 2*row['RAD_SUN']
    # Sun ephemeris
    alpha = np.radians(row['ALPHA'] / 60 / 60)
    P_angle = np.radians(row['P_ANGLE'])
    B_0 = np.radians(row['B_0'])
    # L_0 = np.radians(row['L_0'])
    L_0 = row['L_0']
    
    
    # Relative coordinates
    X = (row['POS_X'] - row['SUN_X']) / (0.5*R_sun)
    Y = (row['POS_Y'] - row['SUN_Y']) / (0.5*R_sun)
    
    # Position angle Pm
    P_m = np.arctan(X/Y)
    # Distance sunspot
    R_m = np.sqrt(X**2 + Y**2)
    
    theta = np.arcsin(R_m)
    
    sin_B = np.sin(B_0)*np.cos(theta) + (Y*np.cos(B_0)*np.sin(theta)) / R_m
    B = np.arcsin(sin_B)
    B = np.rad2deg(B)
    
    sin_L = X*np.sin(theta) / (R_m*np.cos(np.radians(B)))
    
    L = np.arcsin(sin_L)
    # L = np.rad2deg(L) + L_0
    
    # print(np.rad2deg(P_m))
    # visual angle rho
    # rho = np.arcsin(R_m/R_sun) - (alpha/2)*(R_m/R_sun)
    # # print(np.rad2deg(rho))
    # sin_B = np.sin(B_0)*np.cos(rho) + np.cos(B_0)*np.sin(rho)*np.cos(P_angle-P_m)
    # B = np.arcsin(sin_B)
    # B = np.rad2deg(B)
    
    # sin_L_L0 = np.sin(P_angle-P_m)*np.sin(rho)*np.cos(B)
    # L = np.arcsin(sin_L_L0) + L_0
    # L = np.rad2deg(L)
    
    
    
        
    return pd.Series({'LATITUDE': B, 'LONGITUDE': L})
# else:
#     return pd.Series({'Latitude': np.nan, 'Longitude': np.nan})

In [342]:
def helio_coordinates(row):

    R = row['RAD_SUN']

    P = np.radians(row['P_ANGLE'])
    B0 = np.radians(row['B_0'])
    L0 = np.radians(row['L_0'])

    # signed coordinates
    X = (row['POS_X'] - row['SUN_X']) / R
    Y = (row['POS_Y'] - row['SUN_Y']) / R  # flip image Y

    # rotate by P-angle
    x =  X*np.cos(P) - Y*np.sin(P)
    y =  X*np.sin(P) + Y*np.cos(P)

    rho = np.sqrt(x**2 + y**2)
    theta = np.arcsin(rho)
    
    # sin_theta = rho
    # cos_theta = np.sqrt(1 - rho**2)


    # latitude
    sinB = np.sin(B0)*np.cos(theta) + \
           (y*np.sin(theta)*np.cos(B0))/rho
    # sinB = (np.sin(B0)*cos_theta +
    #     (y*sin_theta*np.cos(B0))/rho)

    B = np.arcsin(sinB)

    # central meridian distance
    
    # CMD = np.arctan2(
    # x*sin_theta,
    # rho*cos_theta*np.cos(B0) -
    # y*sin_theta*np.sin(B0)
    # )

    CMD = np.arctan2(
        x*np.sin(theta),
        rho*np.cos(theta)*np.cos(B0) - y*np.sin(theta)*np.sin(B0)
    )

    # Carrington longitude
    # L = -L0 + CMD
    
    # print(L0, CMD, L)


    return pd.Series({
        'LATITUDE': np.degrees(B),
        'LONGITUDE': np.degrees(CMD)
    })


In [276]:

def helio_coordinates_approx(row):
   # Small angle approximation for the latitude
    # small_angle = False

    # x_norm = (row['POS_X'] - row['SUN_X']) / row['RAD_SUN']
    y_norm = (row['POS_Y'] - row['SUN_Y']) / row['RAD_SUN']
    
    B_0 = row['B_0']
    L_0 = row['L_0']
    # Point in the solar disk 
    # if x_norm**2 + y_norm**2 <= 1.0:
        # Latitude
    B = np.degrees(np.arcsin(y_norm))
    # print(B)
    # B = B + B_0
    
    # x_norm = (row['POS_X'] - row['SUN_X']) / row['RAD_SUN']
    
    # print(np.degrees(np.arcsin(x_norm)))
    
    
    x_norm = (row['POS_X']- row['SUN_X']) / row['RAD_SUN']
    
    L = np.degrees(np.arcsin(x_norm) * np.cos(np.radians(B)) )
    
    
   
        # Longitude
        # if small_angle:
    
    # L = -L + L_0
        # else :
        #     lon = np.degrees(np.arcsin(x_norm / np.cos(np.radians(lat))))
        
    
    return pd.Series({'LATITUDE': B, 'LONGITUDE': L})
    # else:
    
    #     return pd.Series({'Latitude': np.nan, 'Longitude': np.nan})

    
    
    
        
    # return pd.Series({'LATITUDE': B, 'LONGITUDE': L})
# else:
#     return pd.Series({'Latitude': np.nan, 'Longitude': np.nan})

In [335]:
def helio_coordinates_new(row):

    R = row['RAD_SUN']

    P = np.radians(row['P_ANGLE'])
    B_0 = np.radians(row['B_0'])
    L_0 = np.radians(row['L_0'])

    # signed coordinates
    X = (row['POS_X'] - row['SUN_X']) #/ R
    Y = (row['POS_Y'] - row['SUN_Y']) #/ R  # flip image Y
    
    ST = np.sqrt(R**2 - (X**2 + Y*22))
    
    sin_B = (Y*np.cos(P) + ST*np.sin(P)) / R
    B = np.arcsin(sin_B)
    # B = B + B_0
    
    sin_L = (X/R) * np.cos(B)
    L = np.arcsin(sin_L)
    # L = L + L_0
    
    
    return pd.Series({'LATITUDE': np.degrees(B), 'LONGITUDE': np.degrees(L)})
    
    

In [350]:
data_09_01_05[['LATITUDE', 'LONGITUDE']] = data_09_01_05.apply(helio_coordinates, axis=1)

# Print table
print(data_09_01_05[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])


                          OBS_TIME  POS_X  POS_Y  LATITUDE  LONGITUDE
0 2005-01-09 06:23:32.198000+00:00     75    469 -6.799331  -1.095892
1 2005-01-09 11:11:32.200000+00:00     87    471 -6.755897  -1.044901
2 2005-01-09 15:59:32.202000+00:00     99    472 -6.811106  -0.997903
3 2005-01-09 23:59:32.206000+00:00    121    476 -6.622650  -0.919811


In [351]:
data_09_01_05[['LATITUDE', 'LONGITUDE']] =data_09_01_05.apply(helio_coordinates_approx, axis=1)

print(data_09_01_05[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])

                          OBS_TIME  POS_X  POS_Y  LATITUDE  LONGITUDE
0 2005-01-09 06:23:32.198000+00:00     75    469 -4.972722 -61.782941
1 2005-01-09 11:11:32.200000+00:00     87    471 -4.741405 -58.987340
2 2005-01-09 15:59:32.202000+00:00     99    472 -4.625793 -56.369774
3 2005-01-09 23:59:32.206000+00:00    121    476 -4.160509 -52.064668


In [352]:
data_09_01_05[['LATITUDE', 'LONGITUDE']] =data_09_01_05.apply(helio_coordinates_new, axis=1)

print(data_09_01_05[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])

                          OBS_TIME  POS_X  POS_Y  LATITUDE  LONGITUDE
0 2005-01-09 06:23:32.198000+00:00     75    469 -5.919923 -61.446621
1 2005-01-09 11:11:32.200000+00:00     87    471 -5.823133 -58.697663
2 2005-01-09 15:59:32.202000+00:00     99    472 -5.841324 -56.106231
3 2005-01-09 23:59:32.206000+00:00    121    476 -5.608323 -51.850023


### Calculating Angular Velocity, Synodic Period, and Sidereal Period

#### Angular Velocity

Angular velocity ($\omega$) is a measure of the rate of rotation and is typically expressed in radians per second (rad/s). It can be calculated using the formula:

$$
\omega = \frac{2\pi}{T}
$$

where $T$ is the period of rotation in seconds.

The angular velocity:

$$
\omega = \frac{\Delta \theta}{\Delta t}
$$

where $\Delta \theta$ is the change in angle in longitude (approximation) and $\Delta t$ is the change in time.

#### Sidereal Period

The sidereal period ($T_{sid}$) is the time taken for a celestial object to complete one full orbit around a central body (e.g., the Sun) with respect to distant stars. It can be calculated simply as:

$$
T_{sid} = \text{time period for one complete revolution}
$$

In the case of planets, this period can be related to their synodic period using the following formula:

$$
\frac{1}{T_{sid}} = \frac{1}{T_s} + \frac{1}{T_{Earth}}
$$

where:
- $T_{sid}$ is the sidereal period,
- $T_s$ is the synodic period,
- $T_{Earth}$ is the orbital period of the Earth (approximately 365.25 days).




In [257]:
# Synodic and Sidereal period


def velocity_and_period(df):
    results = []
    for i in range(len(df)):
        if i + 1 < len(df):
            time_1 = df.iloc[i]['OBS_TIME']
            time_2 = df.iloc[i + 1]['OBS_TIME']
            lon_1 = df.iloc[i]['LONGITUDE']
            lon_2 = df.iloc[i + 1]['LONGITUDE']
            lat_1 = df.iloc[i]['LATITUDE']  # Assumed latitude of the first spot
            
            time_diff_days = (time_2 - time_1).total_seconds() / (24 * 3600)
            
            angular_distance = abs(lon_2 - lon_1)
            
            
            
            angular_velocity = angular_distance / time_diff_days
            
            
            # Synodic period (days)
            synodic_period = 360 / angular_velocity
            
            # Sidereal period using the synodic formula
            sidereal_period_synodic = 1 / (1/synodic_period + 1/360)
            
            # Sidereal period using the theoretical solar model
            lat_radians = np.radians(lat_1)
            sidereal_period_theoretical = 25.38 + 2.77 * np.sin(lat_radians)**2
            
    
            
            results.append({
                'Assumed Latitude': lat_1,
                'Angular Distance (degrees)': angular_distance,
                'Angular Velocity (deg/day)': angular_velocity,
                'Synodic Period (days)': synodic_period,
                'Sidereal Period (Synodic formula)': sidereal_period_synodic,
                'Sidereal Period (Theoretical model)': sidereal_period_theoretical
            })
    
    return pd.DataFrame(results)


In [258]:

ang_vel_09_01 = velocity_and_period(data_09_01_05)

print(ang_vel_09_01)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0         -4.972722                    2.857531                   14.287652   
1         -4.741405                    2.654336                   13.271678   
2         -4.625793                    4.398639                   13.195915   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              25.196582                          23.548416   
1              27.125431                          25.224784   
2              27.281171                          25.359409   

   Sidereal Period (Theoretical model)  
0                            25.400813  
1                            25.398926  
2                            25.398016  


In [307]:
data_17_01_05 = {
    'OBS_TIME': ['2005-01-17T06:23:32.287Z', '2005-01-17T11:11:32.289Z', '2005-01-17T15:59:32.292Z'], # observation time
    'POS_X': [666, 688, 712], # x coordinate of the sunspot 
    'POS_Y': [646, 646, 637], # y coordinate of the sunspot
    'SUN_X': [5.128960571289E+02, 5.128827819824E+02,5.128643493652E+02], # x coordinate of the solar centre
    'SUN_Y': [5.119730834961E+02, 5.119662170410E+02, 5.119624023438E+02], # y coordinate of the solar centre
    'RAD_SUN': [4.955751029422E+02, 4.955711294998E+02, 4.955670897308E+02], # radius of the sun
    # Solar ephemeris
    'ALPHA': [1951.6, 1951.58, 1951.55], # angular diameter of the Sun arcsec
    'P_ANGLE': [-5.775, -5.868, -5.960], #position angle of the rotation axis
    'B_0': [-4.789, -4.808, -4.827], # latitude of the solar disk centre
    'L_0': [159.673, 157.040, 154.406] # longitude of the solar disk centre
}


data_17_01_05 = pd.DataFrame(data_17_01_05)

data_17_01_05['OBS_TIME'] = pd.to_datetime(data_17_01_05['OBS_TIME'])


print(data_17_01_05)

                          OBS_TIME  POS_X  POS_Y       SUN_X       SUN_Y  \
0 2005-01-17 06:23:32.287000+00:00    666    646  512.896057  511.973083   
1 2005-01-17 11:11:32.289000+00:00    688    646  512.882782  511.966217   
2 2005-01-17 15:59:32.292000+00:00    712    637  512.864349  511.962402   

      RAD_SUN    ALPHA  P_ANGLE    B_0      L_0  
0  495.575103  1951.60   -5.775 -4.789  159.673  
1  495.571129  1951.58   -5.868 -4.808  157.040  
2  495.567090  1951.55   -5.960 -4.827  154.406  


In [347]:
data_17_01_05[['LATITUDE', 'LONGITUDE']] = data_17_01_05.apply(helio_coordinates, axis=1)

# Print table
print(data_17_01_05[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])


                          OBS_TIME  POS_X  POS_Y  LATITUDE  LONGITUDE
0 2005-01-17 06:23:32.287000+00:00    666    646  9.266849  19.816655
1 2005-01-17 11:11:32.289000+00:00    688    646  9.035041  22.577192
2 2005-01-17 15:59:32.292000+00:00    712    637  7.724509  25.452341


In [348]:

ang_vel_17_01 = velocity_and_period(data_17_01_05)

print(ang_vel_17_01)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0          9.266849                    2.760537                   13.802683   
1          9.035041                    2.875149                   14.375745   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              26.081887                          24.319916   
1              25.042181                          23.413500   

   Sidereal Period (Theoretical model)  
0                            25.451830  
1                            25.448311  


In [325]:
data_27_12_09 = {
    'OBS_TIME': ['2009-12-27T06:23:31.882Z', '2009-12-27T11:32:31.887Z', '2009-12-27T15:59:31.892Z'], # observation time
    'POS_X': [219, 236, 250], # x coordinate of the sunspot 
    'POS_Y': [293, 293, 294], # y coordinate of the sunspot
    'SUN_X': [5.127313232422E+02, 5.127303771973E+02, 5.127270202637E+02], # x coordinate of the solar centre
    'SUN_Y': [5.116305541992E+02, 5.116247329712E+02, 5.116341857910E+02], # y coordinate of the solar centre
    'RAD_SUN': [4.963766681838E+02, 4.963869073162E+02, 4.963956568728E+02], # radius of the sun
    # Solar ephemeris
    'ALPHA': [1952.33, 1952.35, 1952.37], # angular diameter of the Sun arcsec
    'P_ANGLE': [4.389, 4.288, 4.199], #position angle of the rotation axis
    'B_0': [-2.442, -2.468, -2.490], # latitude of the solar disk centre
    'L_0': [95.160, 92.407, 89.965] # longitude of the solar disk centre
}


data_27_12_09 = pd.DataFrame(data_27_12_09)

data_27_12_09['OBS_TIME'] = pd.to_datetime(data_27_12_09['OBS_TIME'])


print(data_27_12_09)

                          OBS_TIME  POS_X  POS_Y       SUN_X       SUN_Y  \
0 2009-12-27 06:23:31.882000+00:00    219    293  512.731323  511.630554   
1 2009-12-27 11:32:31.887000+00:00    236    293  512.730377  511.624733   
2 2009-12-27 15:59:31.892000+00:00    250    294  512.727020  511.634186   

      RAD_SUN    ALPHA  P_ANGLE    B_0     L_0  
0  496.376668  1952.33    4.389 -2.442  95.160  
1  496.386907  1952.35    4.288 -2.468  92.407  
2  496.395657  1952.37    4.199 -2.490  89.965  


In [345]:
data_27_12_09[['LATITUDE', 'LONGITUDE']] =data_27_12_09.apply(helio_coordinates, axis=1)

print(data_27_12_09[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])

                          OBS_TIME  POS_X  POS_Y   LATITUDE  LONGITUDE
0 2009-12-27 06:23:31.882000+00:00    219    293 -30.848753 -40.389687
1 2009-12-27 11:32:31.887000+00:00    236    293 -30.713126 -37.468424
2 2009-12-27 15:59:31.892000+00:00    250    294 -30.471817 -35.113148


In [346]:

ang_vel_27_12 = velocity_and_period(data_27_12_09)

print(ang_vel_27_12)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0        -30.848753                    2.921263                   13.613651   
1        -30.713126                    2.355276                   12.702608   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              26.444045                          24.634501   
1              28.340637                          26.272371   

   Sidereal Period (Theoretical model)  
0                            26.108335  
1                            26.102569  


In [329]:
data_02_01_2010 = {
    'OBS_TIME': ['2010-01-02T07:58:32.042Z', '2010-01-02T12:07:32.047Z', '2010-01-02T18:41:32.055Z', '2010-01-02T23:59:32.061Z'], # observation time
    'POS_X': [810, 822, 840, 855], # x coordinate of the sunspot 
    'POS_Y': [298, 298, 297, 296], # y coordinate of the sunspot
    'SUN_X': [5.127382532756E+02, 5.127405293783E+02, 5.127350158691E+02, 5.127363891602E+02], # x coordinate of the solar centre
    'SUN_Y': [5.116251195272E+02, 5.116242980957E+02, 5.116033630371E+02, 5.116225280762E+02], # y coordinate of the solar centre
    'RAD_SUN': [4.966171648737E+02, 4.966224656774E+02, 4.966306693200E+02, 4.966371254274E+02], # radius of the sun
    # Solar ephemeris
    'ALPHA': [1952.63, 1952.63, 1952.63, 1952.63], # angular diameter of the Sun arcsec
    'P_ANGLE': [1.456, 1.372, 1.239, 1.132], #position angle of the rotation axis
    'B_0': [-3.161, -3.181, -3.213, -3.238 ], # latitude of the solar disk centre
    'L_0': [15.337, 13.059, 9.456, 6.547] # longitude of the solar disk centre
}


data_02_01_2010 = pd.DataFrame(data_02_01_2010)

data_02_01_2010['OBS_TIME'] = pd.to_datetime(data_02_01_2010['OBS_TIME'])


print(data_02_01_2010)

                          OBS_TIME  POS_X  POS_Y       SUN_X       SUN_Y  \
0 2010-01-02 07:58:32.042000+00:00    810    298  512.738253  511.625120   
1 2010-01-02 12:07:32.047000+00:00    822    298  512.740529  511.624298   
2 2010-01-02 18:41:32.055000+00:00    840    297  512.735016  511.603363   
3 2010-01-02 23:59:32.061000+00:00    855    296  512.736389  511.622528   

      RAD_SUN    ALPHA  P_ANGLE    B_0     L_0  
0  496.617165  1952.63    1.456 -3.161  15.337  
1  496.622466  1952.63    1.372 -3.181  13.059  
2  496.630669  1952.63    1.239 -3.213   9.456  
3  496.637125  1952.63    1.132 -3.238   6.547  


In [343]:
data_02_01_2010[['LATITUDE', 'LONGITUDE']] =data_02_01_2010.apply(helio_coordinates, axis=1)

print(data_02_01_2010[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])

                          OBS_TIME  POS_X  POS_Y   LATITUDE  LONGITUDE
0 2010-01-02 07:58:32.042000+00:00    810    298 -26.836416  43.067074
1 2010-01-02 12:07:32.047000+00:00    822    298 -26.791754  45.149704
2 2010-01-02 18:41:32.055000+00:00    840    297 -26.847559  48.493549
3 2010-01-02 23:59:32.061000+00:00    855    296 -26.908197  51.471913


In [344]:

ang_vel_02_01 = velocity_and_period(data_02_01_2010)

print(ang_vel_02_01)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0        -26.836416                    2.082630                   12.044120   
1        -26.791754                    3.343845                   12.221155   
2        -26.847559                    2.978364                   13.486929   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              29.890105                          27.598643   
1              29.457117                          27.229088   
2              26.692511                          24.849987   

   Sidereal Period (Theoretical model)  
0                            25.944533  
1                            25.942794  
2                            25.944967  
